# Experiment: Sunday Work Expression Classification (Heavy Only)
Pipeline: expression_data.csv -> IgLM heavy embedding -> small classifier on pooled train/test split.


## 1) Setup

In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import transformers

from iglm.model.IgLM import CHECKPOINT_DICT, VOCAB_FILE

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "PDW_GeneralAntibodies").exists():
        return cwd / "PDW_GeneralAntibodies"
    if cwd.name == "PDW_GeneralAntibodies":
        return cwd
    for parent in [cwd, *cwd.parents]:
        cand = parent / "PDW_GeneralAntibodies"
        if cand.exists():
            return cand
    raise FileNotFoundError("Could not locate PDW_GeneralAntibodies")


ROOT = resolve_project_root()

@dataclass
class CFG:
    data_csv: Path = ROOT / "data" / "raw" / "expression_data.csv"
    cache_npz: Path = ROOT / "sunday_work" / "iglm_expression_hl_cache.npz"
    cache_map: Path = ROOT / "sunday_work" / "iglm_expression_hl_cache_map.json"
    recompute_embeddings: bool = False

    model_name: str = "IgLM"
    species_token: str = "[HUMAN]"

    val_frac: float = 0.2
    seed: int = 42

    hidden_dim: int = 64
    dropout: float = 0.3
    lr: float = 1e-3
    weight_decay: float = 2e-3
    batch_size: int = 128
    max_epochs: int = 120
    patience: int = 20

cfg = CFG()
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

print("Using:", cfg.data_csv)
print(cfg)


## 2) Load expression data (already labeled 0/1, pooled)


In [ ]:
df = pd.read_csv(cfg.data_csv)
req = {"heavy", "expression_label", "dataset"}
miss = req - set(df.columns)
if miss:
    raise ValueError(f"Missing columns: {sorted(miss)}")

df["heavy"] = df["heavy"].fillna("").astype(str).str.strip().str.upper()
df["label"] = pd.to_numeric(df["expression_label"], errors="coerce")
df = df[df["label"].isin([0, 1])].copy()
df["source_file"] = df["dataset"].astype(str)

df = df[(df["heavy"] != "")].reset_index(drop=True)

print("rows:", len(df), "studies:", df["source_file"].nunique())
print("class counts:")
print(df["label"].value_counts().to_string())


## 3) IgLM embedding helpers (heavy only, cached)


In [ ]:
def load_vocab() -> dict[str, int]:
    tok2id = {}
    with open(VOCAB_FILE, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            tok = line.strip()
            if tok:
                tok2id[tok] = i
    return tok2id


def embed_heavy(model, device, tok2id, seq: str, species_token: str) -> np.ndarray:
    tokens = ["[HEAVY]", species_token] + list(seq) + ["[SEP]"]
    unk_id = tok2id.get("[UNK]", 1)
    ids = [tok2id.get(t, unk_id) for t in tokens]
    if any(i == unk_id for i in ids):
        raise ValueError("Unknown token")
    x = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        out = model(x, output_hidden_states=True, return_dict=True)
    hs = out.hidden_states[-1][0]
    aa = hs[2:-1]
    if aa.shape[0] == 0:
        raise ValueError("Empty seq")
    return aa.mean(dim=0).cpu().numpy().astype(np.float32)


def load_cache(cfg: CFG) -> dict[str, np.ndarray]:
    if cfg.cache_npz.exists() and cfg.cache_map.exists() and not cfg.recompute_embeddings:
        npz = np.load(cfg.cache_npz, allow_pickle=True)
        key2seq = json.loads(cfg.cache_map.read_text())
        return {seq_key: np.asarray(npz[k], dtype=np.float32) for k, seq_key in key2seq.items()}
    return {}


def save_cache(seq2emb: dict[str, np.ndarray], cfg: CFG) -> None:
    payload, key2seq = {}, {}
    for j, (seq_key, emb) in enumerate(seq2emb.items()):
        k = f"emb_{j:07d}"
        payload[k] = np.asarray(emb, dtype=np.float32)
        key2seq[k] = seq_key
    cfg.cache_npz.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cfg.cache_npz, **payload)
    cfg.cache_map.write_text(json.dumps(key2seq))


def compute_embeddings_for_df(df_in: pd.DataFrame, cfg: CFG) -> dict[str, np.ndarray]:
    need = {f"H::{s}" for s in df_in["heavy"].drop_duplicates().tolist()}

    seq2emb = load_cache(cfg)
    missing = [k for k in need if k not in seq2emb]
    print(f"Cache status: have={len(seq2emb)} need={len(need)} missing={len(missing)}")

    if not missing:
        return seq2emb

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = transformers.GPT2LMHeadModel.from_pretrained(CHECKPOINT_DICT[cfg.model_name]).to(device)
    model.eval()
    tok2id = load_vocab()

    fails = 0
    for i, key in enumerate(missing, start=1):
        seq = key.split("::", 1)[1]
        try:
            seq2emb[key] = embed_heavy(model, device, tok2id, seq, cfg.species_token)
        except Exception:
            fails += 1
        if i % 100 == 0 or i == len(missing):
            print(f"embedded {i}/{len(missing)} (fails={fails})")

    save_cache(seq2emb, cfg)
    return seq2emb


def build_Xy(df_in: pd.DataFrame, seq2emb: dict[str, np.ndarray], label_col: str = "label"):
    rows = []
    for _, r in df_in.iterrows():
        h = seq2emb.get(f"H::{r['heavy']}")
        if h is None:
            continue
        rows.append((h.astype(np.float32), int(r[label_col])))

    X = np.stack([t[0] for t in rows])
    y = np.asarray([t[1] for t in rows], dtype=np.float32)
    return X, y


## 4) Classifier helper (small MLP, BCEWithLogits)

In [ ]:
class ClassifierMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_eval_binary(X_tr, y_tr, X_va, y_va, cfg: CFG):
    # Train-only feature scaling.
    x_mean = X_tr.mean(axis=0, keepdims=True)
    x_std = X_tr.std(axis=0, keepdims=True)
    x_std[x_std < 1e-8] = 1.0

    X_tr_s = (X_tr - x_mean) / x_std
    X_va_s = (X_va - x_mean) / x_std

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ClassifierMLP(X_tr.shape[1], cfg.hidden_dim, cfg.dropout).to(device)

    ds = TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True)

    va_x = torch.tensor(X_va_s, dtype=torch.float32, device=device)
    va_y = torch.tensor(y_va, dtype=torch.float32, device=device)

    pos = float((y_tr == 1).sum())
    neg = float((y_tr == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_state, best_val, bad_epochs = None, float("inf"), 0
    tr_curve, va_curve = [], []

    for _ in range(cfg.max_epochs):
        model.train()
        b_losses = []
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            b_losses.append(float(loss.detach().cpu()))

        tr_loss = float(np.mean(b_losses))
        model.eval()
        with torch.no_grad():
            va_loss = float(loss_fn(model(va_x), va_y).detach().cpu())

        tr_curve.append(tr_loss)
        va_curve.append(va_loss)

        if va_loss < best_val - 1e-6:
            best_val = va_loss
            bad_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.patience:
                break

    model.load_state_dict(best_state)

    with torch.no_grad():
        logits = model(torch.tensor(X_va_s, dtype=torch.float32, device=device)).cpu().numpy()
    prob = 1.0 / (1.0 + np.exp(-logits))
    pred = (prob >= 0.5).astype(int)

    metrics = {
        "acc": float(accuracy_score(y_va.astype(int), pred)),
        "f1": float(f1_score(y_va.astype(int), pred, zero_division=0)),
        "auc": float(roc_auc_score(y_va.astype(int), prob)) if len(np.unique(y_va)) > 1 else float("nan"),
        "epochs": len(tr_curve),
    }
    return metrics, tr_curve, va_curve


## 5) Train on 2 largest datasets, test on 1 small dataset


In [ ]:
seq2emb = compute_embeddings_for_df(df, cfg)

counts = df['source_file'].value_counts()
train_datasets = counts.index[:2].tolist()      # two biggest
small_candidates = counts.index[2:].tolist()    # remaining smaller sets

# Pick the smallest by default (last entry in descending counts).
test_dataset = small_candidates[-1] if small_candidates else counts.index[-1]

train_df = df[df['source_file'].isin(train_datasets)].reset_index(drop=True)
test_df = df[df['source_file'].eq(test_dataset)].reset_index(drop=True)

print('train_datasets:', train_datasets)
print('test_dataset:', test_dataset)
print('train rows:', len(train_df), 'test rows:', len(test_df))
print('train class counts:')
print(train_df['label'].value_counts().to_string())
print('test class counts:')
print(test_df['label'].value_counts().to_string())

X_tr, y_tr = build_Xy(train_df, seq2emb, label_col='label')
X_te, y_te = build_Xy(test_df, seq2emb, label_col='label')

metrics_test, tr1, va1 = train_eval_binary(X_tr, y_tr, X_te, y_te, cfg)
print('2-big -> small test metrics:', metrics_test)


## 6) Loss curve (inline)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(tr1, label='train (2 big datasets)')
plt.plot(va1, label='test (small dataset)')
plt.xlabel('epoch')
plt.ylabel('BCEWithLogits loss')
plt.title('Heavy-only: 2-big train -> small test')
plt.legend(); plt.tight_layout(); plt.show()


## 7) In-domain vs Out-of-domain Comparison Figure
This figure compares performance when validating within the training domains vs testing on the held-out small dataset.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# In-domain check: split only within the two training datasets.
X_tr_all, y_tr_all = build_Xy(train_df, seq2emb, label_col='label')
idx = np.arange(len(y_tr_all))
idx_in_tr, idx_in_va = train_test_split(
    idx,
    test_size=0.2,
    random_state=cfg.seed,
    stratify=y_tr_all.astype(int),
)
metrics_in, tr_in, va_in = train_eval_binary(
    X_tr_all[idx_in_tr], y_tr_all[idx_in_tr], X_tr_all[idx_in_va], y_tr_all[idx_in_va], cfg
)

# Recompute probabilities for balanced accuracy / PR-style threshold check.
# (Reuse train_eval_binary internals by training once more and extracting predictions.)
def fit_and_predict_proba(X_tr, y_tr, X_va, y_va, cfg):
    x_mean = X_tr.mean(axis=0, keepdims=True)
    x_std = X_tr.std(axis=0, keepdims=True)
    x_std[x_std < 1e-8] = 1.0
    X_tr_s = (X_tr - x_mean) / x_std
    X_va_s = (X_va - x_mean) / x_std

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ClassifierMLP(X_tr.shape[1], cfg.hidden_dim, cfg.dropout).to(device)

    ds = TensorDataset(torch.tensor(X_tr_s, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True)
    va_x = torch.tensor(X_va_s, dtype=torch.float32, device=device)
    va_y = torch.tensor(y_va, dtype=torch.float32, device=device)

    pos = float((y_tr == 1).sum())
    neg = float((y_tr == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_state, best_val, bad = None, float('inf'), 0
    for _ in range(cfg.max_epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            vloss = float(loss_fn(model(va_x), va_y).detach().cpu())
        if vloss < best_val - 1e-6:
            best_val = vloss
            bad = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience:
                break

    model.load_state_dict(best_state)
    with torch.no_grad():
        logits = model(torch.tensor(X_va_s, dtype=torch.float32, device=device)).cpu().numpy()
    prob = 1.0 / (1.0 + np.exp(-logits))
    pred = (prob >= 0.5).astype(int)
    return prob, pred

# In-domain probs/preds
in_prob, in_pred = fit_and_predict_proba(
    X_tr_all[idx_in_tr], y_tr_all[idx_in_tr], X_tr_all[idx_in_va], y_tr_all[idx_in_va], cfg
)

# Out-of-domain probs/preds (2-big -> small holdout)
out_prob, out_pred = fit_and_predict_proba(X_tr, y_tr, X_te, y_te, cfg)

comparison = {
    'In-domain (within adams+koenig)': {
        'acc': metrics_in['acc'],
        'auc': metrics_in['auc'],
    },
    f'Out-of-domain (test={test_dataset})': {
        'acc': metrics_test['acc'],
        'auc': metrics_test['auc'],
    }
}

labels = ['acc', 'auc']
in_vals = [comparison['In-domain (within adams+koenig)'][k] for k in labels]
out_vals = [comparison[f'Out-of-domain (test={test_dataset})'][k] for k in labels]

x = np.arange(len(labels))
width = 0.36

plt.figure(figsize=(8,4.5))
plt.bar(x - width/2, in_vals, width, label='In-domain')
plt.bar(x + width/2, out_vals, width, label='Out-of-domain')
plt.ylim(0, 1.0)
plt.xticks(x, labels)
plt.ylabel('Score')
plt.title('Generalization Gap: In-domain vs Out-of-domain')
plt.legend()

for i,v in enumerate(in_vals):
    plt.text(i - width/2, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
for i,v in enumerate(out_vals):
    plt.text(i + width/2, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

comparison

